<a href="https://colab.research.google.com/github/PaulMarv/news-data-text-classifiction/blob/main/huffpost_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:

from datasets import load_dataset

# 2) Load (this keeps data in HF internal cache, not your cwd)
agnews = load_dataset("ag_news")  # {'train': 120000, 'test': 7600}
# Now explicitly save to disk so you can "see" files:
import pandas as pd
agnews["train"].to_pandas().to_csv("ag_news_train.csv", index=False)
agnews["test"].to_pandas().to_csv("ag_news_test.csv", index=False)

# 3) Verify files exist
import os, subprocess, textwrap, sys
print(os.listdir("."))


In [ ]:

from datasets import load_dataset
agnews = load_dataset("ag_news")  # {'train':120000, 'test':7600}

train_ds = agnews["train"]
test_ds = agnews["test"]

train_ds[0]


In [ ]:
import tensorflow as tf

train_texts = list(train_ds["text"])
train_labels = list(train_ds["label"])

test_texts = list(test_ds["text"])
test_labels = list(test_ds["label"])

train_tfds = tf.data.Dataset.from_tensor_slices((train_texts, train_labels))
test_tfds  = tf.data.Dataset.from_tensor_slices((test_texts, test_labels))

BATCH_SIZE = 1024
AUTOTUNE = tf.data.AUTOTUNE

train_tfds = train_tfds.shuffle(10000).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
test_tfds  = test_tfds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

In [ ]:
import re, string

def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ')
    return tf.strings.regex_replace(
        stripped_html,
        '[%s]' % re.escape(string.punctuation),
        ''
    )

vocab_size = 10000
sequence_length = 100

vectorize_layer = tf.keras.layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length
)

text_only = train_tfds.map(lambda x, y: x)
vectorize_layer.adapt(text_only)

In [ ]:
EMBED_DIM = 64

model_keras_emb = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(vocab_size, EMBED_DIM, name="embedding_keras"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4)  # 4 classes
])

model_keras_emb.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer="adam",
    metrics=["accuracy"]
)

history_1 = model_keras_emb.fit(
    train_tfds,
    validation_data=test_tfds,
    epochs=10
)

In [ ]:
!pip install gensim kagglehub

import numpy as np
from gensim.models import KeyedVectors
import kagglehub
import os

# Download latest version using kagglehub
download_path = kagglehub.dataset_download("leadbest/googlenewsvectorsnegative300")
file_path = os.path.join(download_path, "GoogleNews-vectors-negative300.bin.gz")

# Load pretrained 300-dimensional Word2Vec
w2v = KeyedVectors.load_word2vec_format(file_path, binary=True)

embedding_dim = 300
vocab = vectorize_layer.get_vocabulary()

embedding_matrix = np.zeros((len(vocab), embedding_dim))

for i, word in enumerate(vocab):
    if word in w2v:
        embedding_matrix[i] = w2v[word]
    else:
        embedding_matrix[i] = np.random.normal(size=(embedding_dim,))

model_w2v = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(
        len(vocab),
        embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=False,
        name="embedding_w2v"
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4)
])

model_w2v.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer="adam",
    metrics=["accuracy"]
)

history_2 = model_w2v.fit(train_tfds, validation_data=test_tfds, epochs=10)

In [ ]:
import numpy as np
import os


!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

glove_path = "glove.6B.300d.txt"


embeddings_index = {}

with open(glove_path, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

embedding_dim = 300
embedding_matrix_glove = np.zeros((len(vocab), embedding_dim))

for i, word in enumerate(vocab):
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix_glove[i] = vector
    else:
        embedding_matrix_glove[i] = np.random.normal(size=(embedding_dim,))

model_glove = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(
        len(vocab),
        embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix_glove),
        trainable=False,
        name="embedding_glove"
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4)
])

model_glove.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer="adam",
    metrics=["accuracy"]
)

history_3 = model_glove.fit(train_tfds, validation_data=test_tfds, epochs=10)

In [ ]:
!pip install -q seaborn scikit-learn wordcloud

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, itertools, collections
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import label_binarize
from wordcloud import WordCloud, STOPWORDS

os.makedirs("figures", exist_ok=True)

# 4 classes for AG News
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# Convert tf.data -> numpy arrays (texts, labels)
def dataset_to_numpy(ds):
    texts, labels = [], []
    for x, y in ds:
        texts.extend(x.numpy().tolist())
        labels.extend(y.numpy().tolist())
    # decode bytes -> str if needed
    texts = [t.decode("utf-8") if isinstance(t, bytes) else t for t in texts]
    labels = np.array(labels, dtype=int)
    return texts, labels

test_texts_np, test_labels_np = dataset_to_numpy(test_tfds)

# For multiclass curves we’ll make one-hot labels
y_true_bin = label_binarize(test_labels_np, classes=[0,1,2,3])

def softmax_logits_to_preds_and_proba(model, texts):
    """
    Runs model(texts) and returns:
      y_pred  -> integer class predictions (shape (N,))
      y_prob  -> softmax probabilities (shape (N, 4))
    """
    # Build a dataset batch for efficiency
    import tensorflow as tf
    ds = tf.data.Dataset.from_tensor_slices(texts).batch(1024)
    logits = []
    for batch in ds:
        logits.append(model(batch, training=False).numpy())
    logits = np.vstack(logits)  # (N, 4)
    # Convert logits -> probabilities
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    y_prob = e / e.sum(axis=1, keepdims=True)
    y_pred = y_prob.argmax(axis=1)
    return y_pred, y_prob

In [ ]:
from sklearn.metrics import accuracy_score

def plot_and_save_confusion(y_true, y_pred, title_prefix, cmap="Blues"):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    acc = accuracy_score(y_true, y_pred)
    plt.figure(figsize=(6.5, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=label_names, yticklabels=label_names)
    plt.title(f"{title_prefix} — Confusion Matrix (Acc={acc:.3f})")
    plt.xlabel("Predicted"); plt.ylabel("True")
    path = f"figures/{title_prefix.replace(' ', '_').lower()}_confusion_matrix.png"
    plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
    print(f"Saved: {path}")

def classification_report_df(y_true, y_pred, title_prefix):
    rep = classification_report(y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    df = pd.DataFrame(rep).T
    path = f"figures/{title_prefix.replace(' ', '_').lower()}_classification_report.csv"
    df.to_csv(path)
    print(f"Saved: {path}")
    display(df.round(3))
    return df

# Keras trainable embedding
y_pred_1, y_prob_1 = softmax_logits_to_preds_and_proba(model_keras_emb, test_texts_np)
plot_and_save_confusion(test_labels_np, y_pred_1, "Keras Embedding")
classification_report_df(test_labels_np, y_pred_1, "Keras Embedding")

# Word2Vec
y_pred_2, y_prob_2 = softmax_logits_to_preds_and_proba(model_w2v, test_texts_np)
plot_and_save_confusion(test_labels_np, y_pred_2, "Word2Vec")
classification_report_df(test_labels_np, y_pred_2, "Word2Vec")

# GloVe
y_pred_3, y_prob_3 = softmax_logits_to_preds_and_proba(model_glove, test_texts_np)
plot_and_save_confusion(test_labels_np, y_pred_3, "GloVe")
classification_report_df(test_labels_np, y_pred_3, "GloVe")

In [ ]:
def plot_history(history, title_prefix):
    hist = pd.DataFrame(history.history)
    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(hist["accuracy"], label="train")
    if "val_accuracy" in hist: plt.plot(hist["val_accuracy"], label="val")
    plt.title(f"{title_prefix} — Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

    plt.subplot(1,2,2)
    plt.plot(hist["loss"], label="train")
    if "val_loss" in hist: plt.plot(hist["val_loss"], label="val")
    plt.title(f"{title_prefix} — Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

    path = f"figures/{title_prefix.replace(' ', '_').lower()}_training_curves.png"
    plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
    print(f"Saved: {path}")

plot_history(history_1, "Keras Embedding")
plot_history(history_2, "Word2Vec")
plot_history(history_3, "GloVe")

In [ ]:
from sklearn.metrics import f1_score

summary = pd.DataFrame({
    "Model": ["Keras Embedding", "Word2Vec", "GloVe"],
    "Accuracy": [
        accuracy_score(test_labels_np, y_pred_1),
        accuracy_score(test_labels_np, y_pred_2),
        accuracy_score(test_labels_np, y_pred_3)
    ],
    "Macro_F1": [
        f1_score(test_labels_np, y_pred_1, average="macro"),
        f1_score(test_labels_np, y_pred_2, average="macro"),
        f1_score(test_labels_np, y_pred_3, average="macro")
    ]
}).sort_values("Macro_F1", ascending=False)

display(summary.round(4))
summary.to_csv("figures/model_comparison_summary.csv", index=False)
print("Saved: figures/model_comparison_summary.csv")

In [ ]:
# Vectorizer vocabulary is ordered by frequency (index 0 is padding)
vocab = vectorize_layer.get_vocabulary()
top_n = 30  # change as you like
top_tokens = vocab[1:top_n+1]  # skip padding token at index 0

plt.figure(figsize=(10,4.5))
sns.barplot(x=list(range(1, top_n+1)), y=[1]*top_n, hue=None, palette="mako")  # dummy heights
plt.cla()  # we only needed palette
plt.barh(top_tokens[::-1], list(range(1, top_n+1))[::-1], color=sns.color_palette("mako", top_n))
plt.xlabel("Frequency Rank (lower is more frequent)")
plt.title(f"Top {top_n} Tokens by Frequency Rank (Vectorizer Vocabulary)")
path = "figures/top_tokens_by_rank.png"
plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
print(f"Saved: {path}")

In [ ]:
# Build cleaned corpus for wordclouds
extra_stop = set(["rt","https","http","www","com"])  # add anything noisy here
stops = STOPWORDS.union(extra_stop)

def clean_for_wc(text):
    # light normalization; we already lowercase in vectorizer
    text = text.replace("\n", " ")
    return text

# Overall word cloud (test set)
all_text = " ".join(clean_for_wc(t) for t in test_texts_np)
wc = WordCloud(width=1280, height=720, background_color="white",
               stopwords=stops, max_words=300).generate(all_text)
plt.figure(figsize=(12,7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off"); plt.title("AG News — Overall Word Cloud (Test)")
path = "figures/wordcloud_overall.png"
plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
print(f"Saved: {path}")

# Word cloud per class
for cls_idx, cls_name in enumerate(label_names):
    text_c = " ".join(clean_for_wc(t) for t, y in zip(test_texts_np, test_labels_np) if y == cls_idx)
    wc_c = WordCloud(width=1280, height=720, background_color="white",
                     stopwords=stops, max_words=300).generate(text_c)
    plt.figure(figsize=(12,7))
    plt.imshow(wc_c, interpolation="bilinear")
    plt.axis("off"); plt.title(f"AG News — {cls_name} Word Cloud (Test)")
    path = f"figures/wordcloud_{cls_name.lower().replace('/','_')}.png"
    plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
    print(f"Saved: {path}")

In [ ]:
# Compute distribution from test labels (do same for train if you like)
counts = pd.Series(test_labels_np).value_counts().sort_index()
plt.figure(figsize=(5.5,4))
sns.barplot(x=label_names, y=counts.values, palette="viridis")
plt.title("AG News — Class Distribution (Test Split)")
plt.ylabel("Count"); plt.xlabel("Class")
path = "figures/agnews_class_distribution_test.png"
plt.tight_layout(); plt.savefig(path, dpi=200); plt.show()
print(f"Saved: {path}")